# Notebook 04: Feature Engineering
## Purpose: Build 50+ time-series features using PySpark Window Functions
## Input:  workspace.predictive_maintenance.silver_sensor_cleaned
## Output: workspace.predictive_maintenance.silver_sensor_features
## Feature Groups:
##   - Rolling mean + std (10-cycle + 30-cycle windows) = 28 features
##   - Lag features (lag1 + lag5)                       = 14 features
##   - Delta features (rate of change)                  =  7 features
##   - Health index features                            =  2 features
##   Total: 51 engineered features

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType

# Load silver cleaned table
df = spark.table("workspace.predictive_maintenance.silver_sensor_cleaned")

print(f"Loaded: {df.count():,} rows | {len(df.columns)} columns")

# These are our 7 key sensors
SENSORS = [
    'sensor_3', 'sensor_4', 'sensor_7',
    'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12'
]

print(f"Engineering features for: {SENSORS}")

In [0]:
# Window partitioned by machine, ordered by cycle
# This ensures features are calculated PER MACHINE not across all data

w10 = Window.partitionBy("unit_id").orderBy("cycle").rowsBetween(-9, 0)
w30 = Window.partitionBy("unit_id").orderBy("cycle").rowsBetween(-29, 0)
w_lag = Window.partitionBy("unit_id").orderBy("cycle")
w_max = Window.partitionBy("unit_id")

print("Window specs defined:")
print("   w10  — 10-cycle rolling window")
print("   w30  — 30-cycle rolling window")
print("   w_lag — lag features window")
print("   w_max — full machine window (for normalization)")

In [0]:
# 14 features: mean + std for each of 7 sensors over 10 cycles
df_feat = df

for s in SENSORS:
    df_feat = df_feat \
        .withColumn(f"{s}_mean_w10", F.avg(F.col(s)).over(w10)) \
        .withColumn(f"{s}_std_w10",  F.stddev(F.col(s)).over(w10))

# Fill nulls from std (first rows have no std)
df_feat = df_feat.fillna(0)

print(f"   Rolling 10-cycle features added: {len(SENSORS)*2} features")
print(f"   Current column count: {len(df_feat.columns)}")

In [0]:
# 14 more features: mean + std over 30 cycles
for s in SENSORS:
    df_feat = df_feat \
        .withColumn(f"{s}_mean_w30", F.avg(F.col(s)).over(w30)) \
        .withColumn(f"{s}_std_w30",  F.stddev(F.col(s)).over(w30))

df_feat = df_feat.fillna(0)

print(f"   Rolling 30-cycle features added: {len(SENSORS)*2} features")
print(f"   Current column count: {len(df_feat.columns)}")
print(f"   Total rolling features so far: {len(SENSORS)*4}")

In [0]:
# 14 features: lag1 + lag5 for each of 7 sensors
for s in SENSORS:
    df_feat = df_feat \
        .withColumn(f"{s}_lag1", F.lag(F.col(s), 1).over(w_lag)) \
        .withColumn(f"{s}_lag5", F.lag(F.col(s), 5).over(w_lag))

# Drop rows where lag is null (first 5 rows per machine)
rows_before = df_feat.count()
df_feat = df_feat.dropna(subset=[f"{SENSORS[0]}_lag5"])
rows_after = df_feat.count()

print(f"   Lag features added: {len(SENSORS)*2} features")
print(f"   Rows dropped (lag warmup): {rows_before - rows_after:,}")
print(f"   Current column count: {len(df_feat.columns)}")

In [0]:
# 7 features: how fast is each sensor changing cycle-to-cycle?
for s in SENSORS:
    df_feat = df_feat.withColumn(
        f"{s}_delta",
        F.col(s) - F.lag(F.col(s), 1).over(w_lag)
    )

df_feat = df_feat.fillna(0)

print(f"   Delta features added: {len(SENSORS)} features")
print(f"   Current column count: {len(df_feat.columns)}")
print(f"   Total so far: rolling(28) + lag(14) + delta(7) = 49 features")

In [0]:
# Feature 1: normalized_cycle — where is this machine in its lifecycle?
# 0 = brand new, 1 = at failure point
df_feat = df_feat.withColumn(
    "normalized_cycle",
    F.col("cycle") / F.max("cycle").over(w_max)
)

# Feature 2: sensor_deviation_score — composite health score
# Sum of how far each sensor is from its mean (z-score style)
from pyspark.sql.functions import sqrt

deviation_expr = sum([
    F.abs(F.col(s) - F.avg(s).over(w_max)) / 
    (F.stddev(s).over(w_max) + F.lit(1e-6))
    for s in SENSORS
])

df_feat = df_feat.withColumn("sensor_deviation_score", deviation_expr)
df_feat = df_feat.fillna(0)

print(f"   Health index features added: 2")
print(f"   normalized_cycle: 0=new machine → 1=at failure")
print(f"   sensor_deviation_score: composite degradation signal")
print(f"   TOTAL FEATURES: {len(df_feat.columns)} columns")

In [0]:
# List all engineered feature columns
original_cols = list(df.columns)
new_features = [c for c in df_feat.columns if c not in original_cols]

print(f" Original columns:    {len(original_cols)}")
print(f" Engineered features: {len(new_features)}")
print(f" Total columns:       {len(df_feat.columns)}")
print(f"\n=== FEATURE GROUPS ===")
print(f"Rolling mean w10:  {[f for f in new_features if 'mean_w10' in f]}")
print(f"\nRolling std w10:   {[f for f in new_features if 'std_w10' in f]}")
print(f"\nRolling mean w30:  {[f for f in new_features if 'mean_w30' in f]}")
print(f"\nRolling std w30:   {[f for f in new_features if 'std_w30' in f]}")
print(f"\nLag 1:             {[f for f in new_features if 'lag1' in f]}")
print(f"\nLag 5:             {[f for f in new_features if 'lag5' in f]}")
print(f"\nDelta:             {[f for f in new_features if 'delta' in f]}")
print(f"\nHealth Index:      {[f for f in new_features if f in ['normalized_cycle','sensor_deviation_score']]}")

In [0]:
# Write as Delta table with OPTIMIZE
df_feat.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.predictive_maintenance.silver_sensor_features")

# OPTIMIZE for fast ML queries
spark.sql("""
    OPTIMIZE workspace.predictive_maintenance.silver_sensor_features
    ZORDER BY (unit_id, cycle)
""")

# Verify
count = spark.table("workspace.predictive_maintenance.silver_sensor_features").count()
cols  = len(spark.table("workspace.predictive_maintenance.silver_sensor_features").columns)

print(f"   silver_sensor_features written + optimized!")
print(f"   Rows:    {count:,}")
print(f"   Columns: {cols}")

In [0]:
# Show before vs after for one machine
print("=== RAW SENSOR vs ENGINEERED FEATURES (Machine 1) ===")

spark.table("workspace.predictive_maintenance.silver_sensor_features") \
    .filter(F.col("unit_id") == 1) \
    .select(
        "cycle", "sensor_3",
        "sensor_3_mean_w10", "sensor_3_std_w10",
        "sensor_3_lag1", "sensor_3_delta",
        "normalized_cycle", "sensor_deviation_score",
        "RUL", "fail_30"
    ) \
    .orderBy("cycle") \
    .show(10)

## 📋 Feature Engineering Summary

### Why Rolling Windows?
Sensor degradation is gradual — not visible in a single reading.
A 10-cycle window captures short-term trends.
A 30-cycle window captures long-term drift toward failure.

### Why Lag Features?
Lag features tell the model WHERE the sensor was recently.
Combined with current reading, it learns RATE OF CHANGE.

### Why Delta Features?
Delta = current - previous reading.
Sudden spikes in delta = accelerating degradation = failure signal.

### Why Health Index?
normalized_cycle tells the model how far into its life a machine is.
sensor_deviation_score gives a single number for overall machine health.
Lower score = healthy. Rising score = approaching failure.

### Feature Count Breakdown
| Group | Count |
|---|---|
| Rolling mean + std (w10) | 14 |
| Rolling mean + std (w30) | 14 |
| Lag 1 + Lag 5 | 14 |
| Delta (rate of change) | 7 |
| Health Index | 2 |
| **Total Engineered** | **51** |